# Prepare a clean rebuild

Run this notebook once before `04_collect_yearly_data.ipynb`. It archives old collection and analysis outputs under `outputs/rebuild_archives/` so the rebuild starts with empty derived outputs while remaining reversible. It preserves PubMed metadata, abstracts, background-count caches, and completed unified/System 1 decisions.

The notebook checks the configured rebuild slice, currently 2000–2026. The current source vocabularies resolve to 915 pathogens and 245 antibiotics after the configured pathogenic-index filter. This notebook only checks configuration and moves local files; it makes no PubMed or model API calls.

After this notebook, run: `04_collect_yearly_data.ipynb` → `04a_prepare_candidate_abstracts.ipynb` → `04b_predict_candidate_pairs.ipynb` → `04c_build_screened_dataset.ipynb` → `05_model_scientific_interest.ipynb`. The prediction notebook reuses existing decisions and processes candidate rows that are missing from those stores.


In [1]:
from datetime import datetime
from pathlib import Path
import shutil
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
if cfg(PIPELINE_CONFIG, "collection.num_pathogens") is not None:
    raise ValueError("Full rebuild requires collection.num_pathogens: null.")
if cfg(PIPELINE_CONFIG, "collection.num_antibiotics") is not None:
    raise ValueError("Full rebuild requires collection.num_antibiotics: null.")
if cfg(PIPELINE_CONFIG, "collection.search_mode") != "grouped":
    raise ValueError("Full rebuild expects grouped PubMed search mode.")
if int(cfg(PIPELINE_CONFIG, "collection.antibiotic_group_size")) != 10:
    raise ValueError("Full rebuild expects antibiotic groups of 10.")
if not bool(cfg(PIPELINE_CONFIG, "collection.refresh_cached_intervals")):
    raise ValueError("Enable collection.refresh_cached_intervals before rebuilding.")

slice_start = int(cfg(PIPELINE_CONFIG, "collection.slice_start_year"))
slice_end = int(cfg(PIPELINE_CONFIG, "collection.slice_end_year"))
total_end = int(cfg(PIPELINE_CONFIG, "collection.total_end_year"))
if slice_start > slice_end:
    raise ValueError("collection.slice_start_year must be on or before slice_end_year.")
if slice_end != total_end:
    raise ValueError(
        f"Configured rebuild slice ends in {slice_end}, but total_end_year is {total_end}. "
        "Set the slice end to the configured collection end year."
    )

priority_path = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
coarsening_path = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)
priority = pd.read_csv(priority_path)
min_index = float(cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index"))
priority["pathogenic_index"] = pd.to_numeric(priority["pathogenic_index"], errors="coerce")
pathogen_count = priority.loc[
    priority["pathogenic_index"] > min_index, "CanonicalSpecies"
].dropna().astype(str).str.strip().nunique()
antibiotic_count = pd.read_csv(coarsening_path)["Antibiotic"].dropna().astype(str).str.strip().nunique()
print(f"Full collection scope: {pathogen_count:,} pathogens × {antibiotic_count:,} antibiotics")
print(f"Rebuild publication window: {slice_start}-01-01 through {slice_end}-12-31")

parquet_dir = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
output_dir = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
archive_dir = output_dir / "rebuild_archives" / datetime.now().strftime("clean_rebuild_%Y%m%d_%H%M%S")
interval_path = cfg_path(PIPELINE_CONFIG, "paths.interval_pmids", PROJECT_ROOT)
figures_dir = output_dir / "figures"
archive_candidates = [
    interval_path,
    Path(str(interval_path) + ".checkpoints"),
    cfg_path(PIPELINE_CONFIG, "paths.yearly_pair_pmids", PROJECT_ROOT),
    cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT),
    cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT),
    cfg_path(PIPELINE_CONFIG, "paths.group_search_cache", PROJECT_ROOT),
]
analysis_patterns = (
    "yearly_si*.parquet",
    "si_yearly_rank*.parquet",
    "*_smoothed_si_trends.parquet",
    "yearly_si_driver_explanations*.parquet",
)
for pattern in analysis_patterns:
    archive_candidates.extend(parquet_dir.glob(pattern))
if figures_dir.exists():
    archive_candidates.append(figures_dir)

unique_candidates = sorted(
    {path for path in archive_candidates if path.exists()}, key=lambda path: str(path)
)
if not unique_candidates:
    print("No existing collection or analysis outputs found to archive.")
else:
    archive_dir.mkdir(parents=True, exist_ok=False)
    for source in unique_candidates:
        destination = (
            archive_dir / "figures"
            if source == figures_dir
            else archive_dir / source.relative_to(output_dir)
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source), str(destination))
        print(f"Archived {source.relative_to(output_dir)}")
    print(f"Archive location: {archive_dir}")

print(
    "Preserved reusable stores: PubMed metadata, abstract cache, background counts, "
    "unified decisions, System 1 decisions, and unified failure state."
)


Full collection scope: 915 pathogens × 245 antibiotics
Full publication window: 1970-01-01 through 2026-12-31
Archived figures
Archived parquets/candidate_amr_pair_pmid_links.parquet
Archived parquets/candidate_amr_pair_year_counts.parquet
Archived parquets/concave_inverted_u_smoothed_si_trends.parquet
Archived parquets/convex_u_shaped_smoothed_si_trends.parquet
Archived parquets/decreasing_smoothed_si_trends.parquet
Archived parquets/pmids_results.parquet
Archived parquets/pmids_results.parquet.checkpoints
Archived parquets/si_yearly_rank_shift_fit_2000_2023_ref_2025_future_2027.parquet
Archived parquets/yearly_amr_screened_pairs.parquet
Archived parquets/yearly_si.parquet
Archived parquets/yearly_si_class.parquet
Archived parquets/yearly_si_driver_explanations.parquet
Archived parquets/yearly_si_driver_explanations_class.parquet
Archived parquets/yearly_si_driver_explanations_high_level_categories.parquet
Archived parquets/yearly_si_driver_explanations_simplified_categories.parquet
A